# Create Python 3.11 ENV

In [ ]:
!conda create -n femr-py311 python=3.11 -y
!conda run -n femr-py311 python -m pip install ipykernel
!conda run -n femr-py311 python -m pip install torch==2.1.2 \
    --index-url https://download.pytorch.org/whl/cu121
!conda run -n femr-py311 python -m ipykernel install \
    --user \
    --name femr-py311 \
    --display-name "Python 3.11 - FEMR"

!conda run -n femr-py311 python -m pip install femr==0.2.3 datasets==2.15.0 xformers transformers==4.35.2
!conda run -n femr-py311 python -m pip install meds_reader

In [ ]:
!/opt/conda/envs/femr-py311/bin/python3.11 -m pip install --no-cache-dir xformers 

In [ ]:
import sys

!/opt/conda/envs/femr-py311/bin/python3.11 -m pip uninstall -y xformers

!/opt/conda/envs/femr-py311/bin/python3.11 -m pip install \
    --no-cache-dir \
    --no-deps \
    --index-url https://download.pytorch.org/whl/cu121 \
    "xformers==0.0.23.post1"

In [ ]:
!pip install MEDS-Inspect

# Imports

In [ ]:
!/opt/conda/envs/femr-py311/bin/python3.11 -m pip install transformers torch hf-ehr

In [6]:
import os
import re
import pandas as pd
import numpy as np

try:
    from google.cloud import bigquery
except ImportError:
    bigquery = None

import subprocess

cmd = """
source /home/jupyter/load-env.sh >/dev/null
env
"""

result = subprocess.run(
    ["bash", "-lc", cmd],
    capture_output=True,
    text=True,
    check=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

print("WORKSPACE_CDR:", os.environ.get("WORKSPACE_CDR"))
print("WORKSPACE_BUCKET:", os.environ.get("WORKSPACE_BUCKET"))

bucket = os.getenv("WORKSPACE_BUCKET")
cdr = os.environ.get("WORKSPACE_CDR")

if cdr is None:
    raise EnvironmentError(
        "WORKSPACE_CDR is not set. This script should be run inside an All of Us workspace."
    )

use_bqstorage = ("BIGQUERY_STORAGE_API_ENABLED" in os.environ)

WORKSPACE_CDR: wb-silky-artichoke-2408.C2025Q4R6
WORKSPACE_BUCKET: None


# Load Model

In [ ]:
from transformers import AutoModelForCausalLM
import hf_ehr
from hf_ehr.data.tokenization import CLMBRTokenizer
from hf_ehr.config import Event
from typing import List, Dict
import torch

In [ ]:
model = AutoModelForCausalLM.from_pretrained("StanfordShahLab/mamba-tiny-16384-clmbr", trust_remote_code=True) # NOTE: Must trust remote code for Hyena
tokenizer = CLMBRTokenizer.from_pretrained("StanfordShahLab/mamba-tiny-16384-clmbr")

In [ ]:
model

In [ ]:
from transformers import AutoModelForMaskedLM, AutoTokenizer
model_surv = AutoModelForMaskedLM.from_pretrained("boltuix/bert-mini")
tokenizer_surv = AutoTokenizer.from_pretrained("boltuix/bert-mini")

## Full Model Pipeline

### Survey Cross-attn

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SurveyCrossAttention(nn.Module):
    """
    EHR hidden states are queries.
    Survey hidden states are keys/values.

    ehr_hidden_states:
        [batch, ehr_seq_len, ehr_dim]

    survey_hidden_states:
        [batch, survey_seq_len, survey_dim]
    """

    def __init__(
        self,
        ehr_dim: int,
        survey_dim: int,
        num_heads: int = 8,
        dropout: float = 0.1,
        cross_attn_scale_init: float = 0.1,
    ):
        super().__init__()

        if ehr_dim % num_heads != 0:
            raise ValueError(
                f"ehr_dim={ehr_dim} must be divisible by "
                f"num_heads={num_heads}"
            )

        # Project survey representation into Mamba hidden dimension.
        self.survey_proj = nn.Linear(
            survey_dim,
            ehr_dim,
            bias=False,
        )

        # Pre-normalization before cross-attention.
        self.ehr_norm = nn.LayerNorm(ehr_dim)
        self.survey_norm = nn.LayerNorm(ehr_dim)

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=ehr_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.dropout = nn.Dropout(dropout)

        # Start cross-attention as a relatively small perturbation
        # to the pretrained Mamba representation.
        self.cross_attn_scale = nn.Parameter(
            torch.tensor(float(cross_attn_scale_init))
        )

    def forward(
        self,
        ehr_hidden_states,
        survey_hidden_states,
        ehr_attention_mask=None,
        survey_attention_mask=None,
    ):
        """
        Returns:
            [batch, ehr_seq_len, ehr_dim]
        """

        residual = ehr_hidden_states

        # EHR = Q
        q = self.ehr_norm(ehr_hidden_states)

        # Survey = K,V
        survey_hidden_states = self.survey_proj(
            survey_hidden_states
        )
        survey_hidden_states = self.survey_norm(
            survey_hidden_states
        )

        # MultiheadAttention expects True = ignore this key.
        key_padding_mask = None

        if survey_attention_mask is not None:
            key_padding_mask = ~survey_attention_mask.bool()

        attn_output, _ = self.cross_attn(
            query=q,
            key=survey_hidden_states,
            value=survey_hidden_states,
            key_padding_mask=key_padding_mask,
            need_weights=False,
        )

        # Do not add cross-attention output to padded EHR positions.
        if ehr_attention_mask is not None:
            attn_output = (
                attn_output
                * ehr_attention_mask.unsqueeze(-1).to(
                    attn_output.dtype
                )
            )

        hidden_states = (
            residual
            + self.cross_attn_scale
            * self.dropout(attn_output)
        )

        return hidden_states

### Mambda Layer

In [ ]:
class SurveyMambaBlock(nn.Module):
    """
    Pretrained MambaBlock
            ↓
    Survey cross-attention
            ↓
    next MambaBlock
    """

    def __init__(
        self,
        mamba_block,
        ehr_dim: int,
        survey_dim: int,
        num_heads: int = 8,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.mamba_block = mamba_block

        self.survey_cross_attn = SurveyCrossAttention(
            ehr_dim=ehr_dim,
            survey_dim=survey_dim,
            num_heads=num_heads,
            dropout=dropout,
        )

    def forward(
        self,
        hidden_states,
        survey_hidden_states,
        ehr_attention_mask=None,
        survey_attention_mask=None,
    ):
        # Original pretrained Mamba block.
        hidden_states = self.mamba_block(
            hidden_states,
            cache_params=None,
            attention_mask=ehr_attention_mask,
        )

        # Add survey information after the Mamba transformation.
        hidden_states = self.survey_cross_attn(
            ehr_hidden_states=hidden_states,
            survey_hidden_states=survey_hidden_states,
            ehr_attention_mask=ehr_attention_mask,
            survey_attention_mask=survey_attention_mask,
        )

        return hidden_states

### Individual Gene pred

In [ ]:
import torch
import torch.nn as nn


class GeneHeadLayer(nn.Module):
    def __init__(
        self,
        hidden_dim,
        num_selected_layers,
        num_genes,
        gene_embedding_dim=32,
    ):
        super().__init__()

        self.num_genes = num_genes
        self.gene_embedding_dim = gene_embedding_dim

        input_dim = hidden_dim * num_selected_layers

        # Separate projection weights for every gene:
        #
        # gene_weights[g]:
        #   [input_dim, gene_embedding_dim]
        #
        self.gene_weights = nn.Parameter(
            torch.empty(
                num_genes,
                input_dim,
                gene_embedding_dim,
            )
        )

        self.gene_bias = nn.Parameter(
            torch.zeros(
                num_genes,
                gene_embedding_dim,
            )
        )

        # Separate output weights for every gene.
        #
        # output_weights[g]:
        #   [gene_embedding_dim]
        #
        self.output_weights = nn.Parameter(
            torch.empty(
                num_genes,
                gene_embedding_dim,
            )
        )

        self.output_bias = nn.Parameter(
            torch.zeros(num_genes)
        )

        # Initialize each gene's parameters independently.
        nn.init.xavier_uniform_(
            self.gene_weights
        )

        nn.init.xavier_uniform_(
            self.output_weights
        )

    def forward(
        self,
        selected_embeddings,
    ):
        """
        selected_embeddings:
            list of tensors:
                num_selected_layers x [B, hidden_dim]

        returns:
            gene_logits:
                [B, num_genes]
        """

        # [B, num_selected_layers * hidden_dim]
        x = torch.cat(
            selected_embeddings,
            dim=-1,
        )

        # -------------------------------------------------
        # Gene-specific projection
        #
        # x:
        #   [B, D]
        #
        # gene_weights:
        #   [G, D, E]
        #
        # output:
        #   [B, G, E]
        # -------------------------------------------------

        gene_embeddings = torch.einsum(
            "bd,gde->bge",
            x,
            self.gene_weights,
        )

        gene_embeddings = (
            gene_embeddings
            + self.gene_bias.unsqueeze(0)
        )

        gene_embeddings = torch.tanh(
            gene_embeddings
        )

        # -------------------------------------------------
        # Gene-specific classifiers
        #
        # gene_embeddings:
        #   [B, G, E]
        #
        # output_weights:
        #   [G, E]
        #
        # gene_logits:
        #   [B, G]
        # -------------------------------------------------

        gene_logits = torch.einsum(
            "bge,ge->bg",
            gene_embeddings,
            self.output_weights,
        )

        gene_logits = (
            gene_logits
            + self.output_bias.unsqueeze(0)
        )

        return gene_logits

### Germline Pred Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers.modeling_outputs import (
    SequenceClassifierOutput,
)


class GermLinePredModel(nn.Module):
    def __init__(
        self,
        survey_model,
        ehr_model,
        xai_layers,
        num_selected_layers,
        num_genes,
        gene_embedding_dim=32,
        attn_heads=8,
        dropout=0.1,
        pos_weight=None,
    ):
        super().__init__()

        self.xai_layers = list(xai_layers)

        self.num_selected_layers = (
            num_selected_layers
        )

        self.num_genes = num_genes

        self.survey_model = survey_model
        self.ehr_model = ehr_model

        # -----------------------------------------
        # Dimensions
        # -----------------------------------------

        ehr_dim = (
            self.ehr_model.config.hidden_size
        )

        survey_dim = (
            self.survey_model.config.hidden_size
        )

        # -----------------------------------------
        # Survey encoder
        # -----------------------------------------

        self.survey_encoder = (
            self.survey_model.base_model
        )

        # -----------------------------------------
        # Original pretrained Mamba components
        # -----------------------------------------

        self.ehr_embeddings = (
            self.ehr_model.backbone.embeddings
        )

        self.final_norm = (
            self.ehr_model.backbone.norm_f
        )

        # Keep the ORIGINAL pretrained MambaBlock
        # objects and add survey cross-attention
        # around them.
        original_layers = list(
            self.ehr_model.backbone.layers
        )

        self.layers = nn.ModuleList(
            [
                SurveyMambaBlock(
                    mamba_block=layer,
                    ehr_dim=ehr_dim,
                    survey_dim=survey_dim,
                    num_heads=attn_heads,
                    dropout=dropout,
                )
                for layer in original_layers
            ]
        )

        # -----------------------------------------
        # Overall germline-predisposition head
        # -----------------------------------------

        self.prediction_head = nn.Linear(
            ehr_dim,
            1,
        )

        # -----------------------------------------
        # Individual-gene head
        # -----------------------------------------

        self.gene_pred_layer = GeneHeadLayer(
            hidden_dim=ehr_dim,
            num_selected_layers=(
                num_selected_layers
            ),
            num_genes=num_genes,
            gene_embedding_dim=(
                gene_embedding_dim
            ),
        )

        # -----------------------------------------
        # Loss
        #
        # Expected:
        # [overall, gene1, ..., gene73]
        # -----------------------------------------

        if pos_weight is not None:
            self.register_buffer(
                "pos_weight",
                torch.as_tensor(
                    pos_weight,
                    dtype=torch.float32,
                ),
            )

        else:
            self.pos_weight = None

    def forward(
        self,
        ehr_input_ids,
        ehr_attention_mask,
        survey_input_ids,
        survey_attention_mask,
        labels=None,
    ):
        # =========================================
        # Survey sequence
        # =========================================

        survey_outputs = (
            self.survey_encoder(
                input_ids=survey_input_ids,
                attention_mask=(
                    survey_attention_mask
                ),
                return_dict=True,
            )
        )

        survey_hidden_states = (
            survey_outputs.last_hidden_state
        )

        # =========================================
        # EHR embeddings
        # =========================================

        hidden_states = (
            self.ehr_embeddings(
                ehr_input_ids
            )
        )

        # =========================================
        # Last valid event for each patient
        # =========================================

        patient_lengths = (
            ehr_attention_mask
            .long()
            .sum(dim=1)
        )

        final_event_indices = (
            patient_lengths - 1
        )

        batch_indices = torch.arange(
            hidden_states.shape[0],
            device=hidden_states.device,
        )

        # =========================================
        # Mamba layers + survey cross-attention
        # =========================================

        selected_embeddings = []

        for layer_idx, layer in enumerate(
            self.layers
        ):
            hidden_states = layer(
                hidden_states=hidden_states,
                survey_hidden_states=(
                    survey_hidden_states
                ),
                ehr_attention_mask=(
                    ehr_attention_mask
                ),
                survey_attention_mask=(
                    survey_attention_mask
                ),
            )

            # Save patient representation from
            # selected layers for gene prediction.
            if layer_idx in self.xai_layers:
                layer_embedding = (
                    hidden_states[
                        batch_indices,
                        final_event_indices,
                    ]
                )

                selected_embeddings.append(
                    layer_embedding
                )

        # =========================================
        # Final Mamba normalization
        # =========================================

        hidden_states = self.final_norm(
            hidden_states
        )

        # =========================================
        # Final patient embedding
        # =========================================

        patient_embedding = (
            hidden_states[
                batch_indices,
                final_event_indices,
            ]
        )

        # If the final layer is an XAI layer,
        # use its normalized final representation.
        final_layer_idx = (
            len(self.layers) - 1
        )

        if final_layer_idx in self.xai_layers:
            xai_idx = self.xai_layers.index(
                final_layer_idx
            )

            selected_embeddings[
                xai_idx
            ] = patient_embedding

        # =========================================
        # Overall germline predisposition
        # =========================================

        overall_logit = (
            self.prediction_head(
                patient_embedding
            )
        )

        # [B, 1]

        # =========================================
        # Individual genes
        # =========================================

        gene_logits = (
            self.gene_pred_layer(
                selected_embeddings
            )
        )

        # [B, 73]

        # =========================================
        # Full output
        #
        # [overall, gene1, ..., gene73]
        # =========================================

        logits = torch.cat(
            (
                overall_logit,
                gene_logits,
            ),
            dim=1,
        )

        # [B, 74]

        # =========================================
        # Loss over complete 74-dimensional vector
        # =========================================

        loss = None

        if labels is not None:
            labels = labels.float()
        
            # -----------------------------------------
            # Overall germline predisposition
            # -----------------------------------------
        
            overall_labels = labels[:, 0]
            overall_logits_flat = overall_logit.squeeze(-1)
        
            if self.pos_weight is not None:
                overall_pos_weight = self.pos_weight[0]
            else:
                overall_pos_weight = None
        
            overall_loss = F.binary_cross_entropy_with_logits(
                overall_logits_flat,
                overall_labels,
                pos_weight=overall_pos_weight,
            )
        
            # -----------------------------------------
            # Individual genes
            # -----------------------------------------
        
            gene_labels = labels[:, 1:]
        
            if self.pos_weight is not None:
                gene_pos_weight = self.pos_weight[1:]
            else:
                gene_pos_weight = None
        
            gene_loss = F.binary_cross_entropy_with_logits(
                gene_logits,
                gene_labels,
                pos_weight=gene_pos_weight,
            )
        
            # -----------------------------------------
            # Joint multitask objective
            # -----------------------------------------
        
            loss = (
                0.5 * overall_loss
                + 0.5 * gene_loss
            )

        return {
            "loss": loss,
            "logits": logits,
            "overall_loss": overall_loss if labels is not None else None,
            "gene_loss": gene_loss if labels is not None else None,
        }

## Dataset Pipeline

In [ ]:
from hf_ehr.config import Event


def meds_subject_to_hf_events(subject):
    """
    Convert one meds_reader.Subject into one complete
    chronological List[hf_ehr.config.Event].
    """

    patient_events = []

    for meds_event in subject.events:
        properties = dict(meds_event)

        numeric_value = properties.get(
            "numeric_value"
        )

        text_value = properties.get(
            "text_value"
        )

        # Prefer numeric values when available because
        # CLMBR-style tokenization can bin numeric values.
        if numeric_value is not None:
            value = numeric_value
        else:
            value = text_value

        unit = properties.get("unit")

        end = properties.get("end")

        omop_table = properties.get(
            "omop_table"
        )

        event = Event(
            code=str(meds_event.code),
            value=value,
            unit=unit,
            start=meds_event.time,
            end=end,
            omop_table=omop_table,
        )

        patient_events.append(event)

    return patient_events

from torch.utils.data import Dataset


class PatientEHRSurveyDataset(Dataset):
    def __init__(
        self,
        meds_database,
        subject_ids,
        survey_by_subject,
        labels_by_subject,
    ):
        """
        meds_database:
            meds_reader.SubjectDatabase

        subject_ids:
            Patient IDs for this train/val/test split.

        survey_by_subject:
            Mapping:
                subject_id -> survey text

        labels_by_subject:
            For overall germline:
                subject_id -> 0/1

            For gene prediction:
                subject_id -> vector[num_genes]
        """

        self.meds_database = meds_database

        self.subject_ids = [
            int(x)
            for x in subject_ids
        ]

        self.survey_by_subject = (
            survey_by_subject
        )

        self.labels_by_subject = (
            labels_by_subject
        )

    def __len__(self):
        return len(self.subject_ids)

    def __getitem__(self, index):
        subject_id = self.subject_ids[index]

        # Entire longitudinal patient.
        subject = self.meds_database[
            subject_id
        ]

        ehr_events = meds_subject_to_hf_events(
            subject
        )

        survey_text = self.survey_by_subject[
            subject_id
        ]

        label = self.labels_by_subject[
            subject_id
        ]

        return {
            "ehr_events": ehr_events,
            "survey_text": survey_text,
            "label": label,
        }

In [ ]:
from dataclasses import dataclass
from torch.nn.utils.rnn import pad_sequence


@dataclass
class PatientBatchCollator:
    ehr_tokenizer: object
    survey_tokenizer: object

    # Optional safety check.
    # None means never truncate EHR here.
    max_ehr_tokens: int | None = None

    def __call__(self, features):
        # ==============================================
        # EHR
        # ==============================================

        ehr_sequences = []

        for feature in features:
            patient_events = feature[
                "ehr_events"
            ]

            # Tokenize ONE WHOLE patient.
            encoded = self.ehr_tokenizer(
                [patient_events],
                add_special_tokens=True,
                return_tensors="pt",
            )

            ids = encoded[
                "input_ids"
            ].squeeze(0)

            if (
                self.max_ehr_tokens is not None
                and len(ids) > self.max_ehr_tokens
            ):
                raise ValueError(
                    f"Patient has {len(ids)} EHR tokens, "
                    f"which exceeds max_ehr_tokens="
                    f"{self.max_ehr_tokens}. "
                    "The collator will not silently split "
                    "or truncate patients."
                )

            ehr_sequences.append(ids)

        # Right padding.
        ehr_pad_token_id = getattr(
            self.ehr_tokenizer,
            "pad_token_id",
            None,
        )

        # CLMBR tokenizers do not necessarily require a
        # dedicated pad ID. Since these are trailing masked
        # positions, a valid fallback ID can be used.
        if ehr_pad_token_id is None:
            ehr_pad_token_id = 0

        ehr_input_ids = pad_sequence(
            ehr_sequences,
            batch_first=True,
            padding_value=ehr_pad_token_id,
        )

        ehr_lengths = torch.tensor(
            [
                len(sequence)
                for sequence in ehr_sequences
            ],
            dtype=torch.long,
        )

        max_ehr_length = ehr_input_ids.shape[1]

        ehr_attention_mask = (
            torch.arange(max_ehr_length)
            .unsqueeze(0)
            < ehr_lengths.unsqueeze(1)
        ).long()

        # ==============================================
        # SURVEY
        # ==============================================

        survey_texts = [
            feature["survey_text"]
            for feature in features
        ]

        # No splitting.
        # No truncation.
        survey_batch = self.survey_tokenizer(
            survey_texts,
            padding=True,
            truncation=False,
            return_tensors="pt",
        )

        survey_input_ids = survey_batch[
            "input_ids"
        ]

        survey_attention_mask = (
            survey_batch["attention_mask"]
        )

        # ==============================================
        # LABELS
        # ==============================================

        labels = torch.stack(
            [
                torch.as_tensor(
                    feature["label"],
                    dtype=torch.float32,
                )
                for feature in features
            ]
        )

        return {
            "ehr_input_ids": ehr_input_ids,
            "ehr_attention_mask": ehr_attention_mask,
            "survey_input_ids": survey_input_ids,
            "survey_attention_mask":
                survey_attention_mask,
            "labels": labels,
        }

## Deprecated

In [ ]:
from huggingface_hub import notebook_login
import femr.models.transformer
import torch
import femr.models.tokenizer
import femr.models.processor
import datetime

notebook_login()

In [ ]:
model_name = "StanfordShahLab/motor-t-base"

# Load tokenizer / batch loader
# motor_tokenizer = femr.models.tokenizer.FEMRTokenizer.from_pretrained(model_name)
# motor_batch_processor = femr.models.processor.FEMRBatchProcessor(motor_tokenizer)

# Load model
motor_model = femr.models.transformer.FEMRModel.from_pretrained(model_name)

In [ ]:
import inspect
import femr.models.transformer as femr_transformer

print(inspect.signature(
    femr_transformer.FEMREncoderLayer.__init__
))

print(inspect.signature(
    femr_transformer.FEMREncoderLayer.forward
))

print(inspect.signature(
    femr_transformer.FEMRTransformer.forward
))

print(inspect.getsource(
    femr.models.transformer.FEMREncoderLayer
))

print(inspect.getsource(
    femr_transformer.FEMRTransformer
))

In [ ]:
from transformers import AutoModelForMaskedLM, AutoTokenizer
model = AutoModelForMaskedLM.from_pretrained("boltuix/bert-mini")
tokenizer = AutoTokenizer.from_pretrained("boltuix/bert-mini")

### Full Model Pipeline

In [ ]:
import torch
import torch.nn as nn

import torch
import torch.nn as nn
import torch.nn.functional as F
import xformers

from torch.nn.utils.rnn import pad_sequence

import femr.models.transformer
from femr.models.transformer import fixed_pos_embedding

#### Survey Cross attn

In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence


class SurveyCrossAttn(nn.Module):
    def __init__(
        self,
        attn_heads,
        survey_dim,
        ehr_dim,
        dropout=0.1,
    ):
        super().__init__()

        if ehr_dim % attn_heads != 0:
            raise ValueError(
                f"ehr_dim={ehr_dim} must be divisible by "
                f"attn_heads={attn_heads}"
            )

        self.survey_embed_proj = (
            nn.Identity()
            if survey_dim == ehr_dim
            else nn.Linear(survey_dim, ehr_dim)
        )

        self.ehr_norm = femr.models.rmsnorm.RMSNorm(ehr_dim)
        self.survey_norm = femr.models.rmsnorm.RMSNorm(ehr_dim)
        self.output_norm = femr.models.rmsnorm.RMSNorm(ehr_dim)

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=ehr_dim,
            num_heads=attn_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.dropout = nn.Dropout(dropout)

        
    def forward(
        self,
        ehr_hidden,
        survey_hidden,
        patient_lengths,
        survey_attention_mask=None,
    ):
        """
        ehr_hidden:
            [sum(patient_lengths), ehr_dim]

        survey_hidden:
            [batch_size, survey_length, survey_dim]

        patient_lengths:
            [batch_size]

        survey_attention_mask:
            [batch_size, survey_length]
            1 = valid survey token
            0 = padding
        """

        # Keep a tensor on the same device as the EHR hidden states.
        patient_lengths_device = patient_lengths.to(
            device=ehr_hidden.device,
            dtype=torch.long,
        )

        # torch.split requires Python integers.
        patient_lengths_list = (
            patient_lengths_device.detach().cpu().tolist()
        )

        if sum(patient_lengths_list) != ehr_hidden.shape[0]:
            raise ValueError(
                "The sum of patient_lengths must equal the number "
                f"of packed EHR tokens. Got "
                f"{sum(patient_lengths_list)} and "
                f"{ehr_hidden.shape[0]}."
            )

        if survey_hidden.shape[0] != len(patient_lengths_list):
            raise ValueError(
                "The EHR and survey batch sizes do not match. "
                f"Got {len(patient_lengths_list)} EHR sequences "
                f"and {survey_hidden.shape[0]} survey sequences."
            )

        # Split packed EHR states into one sequence per patient.
        ehr_sequences = torch.split(
            ehr_hidden,
            patient_lengths_list,
            dim=0,
        )

        # [batch_size, max_ehr_length, ehr_dim]
        padded_ehr = pad_sequence(
            ehr_sequences,
            batch_first=True,
            padding_value=0.0,
        )

        max_ehr_length = padded_ehr.shape[1]

        # [batch_size, max_ehr_length]
        # True indicates a real EHR position.
        ehr_valid_mask = (
            torch.arange(
                max_ehr_length,
                device=ehr_hidden.device,
            ).unsqueeze(0)
            < patient_lengths_device.unsqueeze(1)
        )

        # [batch_size, survey_length, ehr_dim]
        survey_hidden = self.survey_embed_proj(survey_hidden)

        query = self.ehr_norm(padded_ehr)
        key_value = self.survey_norm(survey_hidden)

        survey_padding_mask = None

        if survey_attention_mask is not None:
            survey_attention_mask = survey_attention_mask.to(
                device=survey_hidden.device
            ).bool()

            if survey_attention_mask.shape != survey_hidden.shape[:2]:
                raise ValueError(
                    "survey_attention_mask must have shape "
                    "[batch_size, survey_length]."
                )

            # Every patient must have at least one survey token.
            if (~survey_attention_mask.any(dim=1)).any():
                raise ValueError(
                    "At least one patient has no valid survey tokens."
                )

            # MultiheadAttention:
            # True means ignore this key/value position.
            survey_padding_mask = ~survey_attention_mask

        # Batched cross-attention:
        #
        # Q = [B, max_EHR_length, ehr_dim]
        # K = [B, survey_length, ehr_dim]
        # V = [B, survey_length, ehr_dim]
        cross_output, _ = self.cross_attention(
            query=query,
            key=key_value,
            value=key_value,
            key_padding_mask=survey_padding_mask,
            need_weights=False,
        )

        # Residual connection around cross-attention.
        output = self.output_norm(
            padded_ehr + self.dropout(cross_output)
        )

        # Discard padded EHR query positions and restore FEMR's
        # packed ordering.
        packed_output = output[ehr_valid_mask]

        return packed_output

#### Updated FEMR Encoder layer

In [ ]:
class FEMREncoderLayer(
    femr.models.transformer.FEMREncoderLayer
):
    def __init__(
        self,
        config,
        attn_heads,
        survey_dim,
        ehr_dim=None,
        dropout=0.1,
    ):
        super().__init__(config)

        if ehr_dim is None:
            ehr_dim = config.hidden_size

        if ehr_dim != config.hidden_size:
            raise ValueError(
                f"ehr_dim={ehr_dim} must match FEMR hidden size "
                f"{config.hidden_size}"
            )

        self.cross_attn = SurveyCrossAttn(
            attn_heads=attn_heads,
            survey_dim=survey_dim,
            ehr_dim=ehr_dim,
            dropout=dropout,
        )

    def forward(
        self,
        x,
        normed_ages,
        pos_embed,
        attn_bias,
        survey_hidden,
        patient_lengths,
        survey_attention_mask=None,
    ):
        # Original FEMR layer returns the update that is
        # normally added in FEMRTransformer.forward().
        x = super().forward(
            x=x,
            normed_ages=normed_ages,
            pos_embed=pos_embed,
            attn_bias=attn_bias,
        )

        # New survey cross-attention.
        
        # Q = EHR hidden states
        # K = survey hidden states
        # V = survey hidden states
        cross_attn = self.cross_attn(
            ehr_hidden=x,
            survey_hidden=survey_hidden,
            patient_lengths=patient_lengths,
            survey_attention_mask=survey_attention_mask,
        )

        return x + cross_attn # FEMR Residual connection

#### Individual Gene prediction layer

In [ ]:
class GeneHeadLayer(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        num_selected_layers: int,
        num_genes: int,
        gene_embedding_dim: int = 32,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.num_genes = num_genes
        self.gene_embedding_dim = gene_embedding_dim

        concatenated_dim = hidden_size * num_selected_layers

        # Normalize each selected transformer representation separately.
        self.layer_norms = nn.ModuleList(
            [
                nn.LayerNorm(hidden_size)
                for _ in range(num_selected_layers)
            ]
        )

        self.dropout = nn.Dropout(dropout)

        # Produce one latent vector for every gene.
        #
        # [B, num_selected_layers * H]
        #                ->
        # [B, num_genes * gene_embedding_dim]
        self.to_gene_embeddings = nn.Linear(
            concatenated_dim,
            num_genes * gene_embedding_dim,
        )

        self.gene_embedding_norm = nn.LayerNorm(
            gene_embedding_dim
        )

        # One classifier vector per gene.
        #
        # gene_classifier_weights[g] is used only for gene g.
        self.gene_classifier_weights = nn.Parameter(
            torch.empty(num_genes, gene_embedding_dim)
        )

        self.gene_classifier_bias = nn.Parameter(
            torch.zeros(num_genes)
        )

        nn.init.xavier_uniform_(
            self.gene_classifier_weights
        )

    def forward(
        self,
        selected_patient_embeddings: list[torch.Tensor],
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        selected_patient_embeddings:
            List containing one [B, H] tensor per selected layer.

        Returns:
            gene_logits:
                [B, num_genes]

            gene_embeddings:
                [B, num_genes, gene_embedding_dim]
        """

        if len(selected_patient_embeddings) != len(
            self.layer_norms
        ):
            raise ValueError(
                "Number of supplied layer outputs does not match "
                "num_selected_layers."
            )

        normalized_outputs = [
            norm(layer_output)
            for norm, layer_output in zip(
                self.layer_norms,
                selected_patient_embeddings,
            )
        ]

        # [B, H] + [B, H] + [B, H]
        #              ->
        # [B, 3H]
        combined = torch.cat(
            normalized_outputs,
            dim=-1,
        )

        combined = self.dropout(combined)

        # [B, 3H] -> [B, G * Dg]
        gene_embeddings = self.to_gene_embeddings(
            combined
        )

        # [B, G * Dg] -> [B, G, Dg]
        gene_embeddings = gene_embeddings.view(
            gene_embeddings.shape[0],
            self.num_genes,
            self.gene_embedding_dim,
        )

        gene_embeddings = self.gene_embedding_norm(
            gene_embeddings
        )

        # For each gene g:
        #
        # logit_g = gene_embedding_g · classifier_weight_g + bias_g
        #
        # [B, G, Dg] * [G, Dg]
        #                 ->
        # [B, G]
        gene_logits = (
            gene_embeddings
            * self.gene_classifier_weights.unsqueeze(0)
        ).sum(dim=-1)

        gene_logits = (
            gene_logits
            + self.gene_classifier_bias.unsqueeze(0)
        )

        return gene_logits, gene_embeddings

#### Model Pipeline

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import xformers.ops


class GermLinePredModel(nn.Module):
    def __init__(
        self,
        survey_model,
        survey_tokenizer,
        ehr_model,
        ehr_tokenizer,
        xai_layers,
        num_selected_layers,
        num_genes,
        gene_embedding_dim: int = 32,
        survey_dim=None,
        ehr_dim=None,
        attn_heads=8,
        dropout=0.1,
        overall_loss_weight: float = 1.0,
        gene_loss_weight: float = 1.0,
    ):
        super().__init__()

        self.survey_embed_model = survey_model
        self.survey_embed_token = survey_tokenizer

        self.ehr_model = ehr_model
        self.ehr_token = ehr_tokenizer

        self.num_genes = num_genes
        self.overall_loss_weight = overall_loss_weight
        self.gene_loss_weight = gene_loss_weight

        ehr_transformer = self.ehr_model.transformer
        config = ehr_transformer.config

        num_transformer_layers = len(
            ehr_transformer.layers
        )

        # Convert negative indices, such as -1, to normal indices.
        normalized_xai_layers = []

        for layer_index in xai_layers:
            if layer_index < 0:
                layer_index = (
                    num_transformer_layers + layer_index
                )

            if not 0 <= layer_index < num_transformer_layers:
                raise ValueError(
                    f"Invalid XAI layer index {layer_index}. "
                    f"The transformer has "
                    f"{num_transformer_layers} layers."
                )

            normalized_xai_layers.append(layer_index)

        # Remove duplicates but preserve the supplied order.
        self.xai_layers = tuple(
            dict.fromkeys(normalized_xai_layers)
        )

        if len(self.xai_layers) != num_selected_layers:
            raise ValueError(
                f"num_selected_layers={num_selected_layers}, "
                f"but {len(self.xai_layers)} unique layer "
                f"indices were provided: {self.xai_layers}"
            )

        if survey_dim is None:
            survey_dim = (
                self.survey_embed_model.config.hidden_size
            )

        if ehr_dim is None:
            ehr_dim = config.hidden_size

        if ehr_dim != config.hidden_size:
            raise ValueError(
                f"ehr_dim={ehr_dim} must match CLMBR hidden "
                f"size {config.hidden_size}"
            )

        # Replace every original FEMR layer with the modified
        # FEMR + survey-cross-attention layer.
        modified_layers = nn.ModuleList()

        for original_layer in ehr_transformer.layers:
            modified_layer = FEMREncoderLayer(
                config=config,
                attn_heads=attn_heads,
                survey_dim=survey_dim,
                ehr_dim=ehr_dim,
                dropout=dropout,
            )

            loading_info = modified_layer.load_state_dict(
                original_layer.state_dict(),
                strict=False,
            )

            # Missing keys should correspond to newly added
            # cross-attention parameters.
            # Unexpected keys should normally be empty.
            if loading_info.unexpected_keys:
                raise RuntimeError(
                    "Unexpected FEMR checkpoint parameters: "
                    f"{loading_info.unexpected_keys}"
                )

            modified_layers.append(modified_layer)

        ehr_transformer.layers = modified_layers

        self.gene_pred_layer = GeneHeadLayer(
            hidden_size=ehr_dim,
            num_selected_layers=num_selected_layers,
            num_genes=num_genes,
            gene_embedding_dim=gene_embedding_dim,
        )

        # One logit for the overall germline-positive task.
        self.overall_classifier = nn.Sequential(
            nn.LayerNorm(ehr_dim),
            nn.Dropout(dropout),
            nn.Linear(ehr_dim, 1),
        )

    def forward(
        self,
        ehr_batch,
        survey_input_ids,
        survey_attention_mask,
        labels=None,
    ):
        # --------------------------------------------------
        # 1. Encode the survey sequences
        # --------------------------------------------------

        survey_outputs = self.survey_embed_model(
            input_ids=survey_input_ids,
            attention_mask=survey_attention_mask,
            return_dict=True,
        )

        survey_hidden = survey_outputs.last_hidden_state

        # survey_hidden:
        # [B, survey_length, survey_dim]

        # --------------------------------------------------
        # 2. Reproduce the FEMR transformer input path
        # --------------------------------------------------

        transformer = self.ehr_model.transformer
        config = transformer.config

        if not config.is_hierarchical:
            x = transformer.embed(
                ehr_batch["tokens"]
            )
        else:
            x = transformer.embed_bag(
                ehr_batch["hierarchical_tokens"],
                ehr_batch["token_indices"],
                ehr_batch["hierarchical_weights"],
            )

        # x:
        # [total_packed_events, ehr_dim]
        x = transformer.in_norm(x)

        patient_lengths = ehr_batch["patient_lengths"]

        if torch.any(patient_lengths <= 0):
            raise ValueError(
                "Every patient must contain at least one "
                "EHR sequence position."
            )

        if patient_lengths.sum().item() != x.shape[0]:
            raise ValueError(
                "The sum of patient_lengths does not match "
                "the packed EHR sequence length."
            )

        # Compute this only once.
        final_event_indices = (
            torch.cumsum(patient_lengths, dim=0) - 1
        ).long()

        normed_ages = ehr_batch["normalized_ages"]

        pos_embed = fixed_pos_embedding(
            ehr_batch["ages"],
            config.hidden_size // config.n_heads,
            x.dtype,
        )

        attn_bias = (
            xformers.ops.fmha.attn_bias.BlockDiagonalMask
            .from_seqlens(patient_lengths.tolist())
            .make_local_attention(
                config.attention_width
            )
        )

        # --------------------------------------------------
        # 3. EHR self-attention and survey cross-attention
        # --------------------------------------------------

        selected_embeddings_by_layer = {}

        final_layer_index = len(transformer.layers) - 1

        for layer_index, layer in enumerate(
            transformer.layers
        ):
            x = layer(
                x=x,
                normed_ages=normed_ages,
                pos_embed=pos_embed,
                attn_bias=attn_bias,
                survey_hidden=survey_hidden,
                patient_lengths=patient_lengths,
                survey_attention_mask=(
                    survey_attention_mask
                ),
            )

            # Save intermediate layers here.
            #
            # The final layer is saved after out_norm below.
            if (
                layer_index in self.xai_layers
                and layer_index != final_layer_index
            ):
                selected_embeddings_by_layer[
                    layer_index
                ] = x[final_event_indices]

        # Apply CLMBR's final normalization.
        x = transformer.out_norm(x)

        # Main patient representation:
        # [B, ehr_dim]
        patient_embeddings = x[final_event_indices]

        # If the final layer is selected, use its normalized
        # patient representation.
        if final_layer_index in self.xai_layers:
            selected_embeddings_by_layer[
                final_layer_index
            ] = patient_embeddings

        # Restore exactly the order specified in self.xai_layers.
        xai_embeddings = [
            selected_embeddings_by_layer[layer_index]
            for layer_index in self.xai_layers
        ]

        # --------------------------------------------------
        # 4. Overall and gene-specific predictions
        # --------------------------------------------------

        overall_logits = self.overall_classifier(
            patient_embeddings
        )

        # overall_logits:
        # [B, 1]

        gene_logits, gene_embeddings = (
            self.gene_pred_layer(xai_embeddings)
        )

        # gene_logits:
        # [B, num_genes]
        #
        # gene_embeddings:
        # [B, num_genes, gene_embedding_dim]

        if gene_logits.shape[-1] != self.num_genes:
            raise RuntimeError(
                f"Gene head returned {gene_logits.shape[-1]} "
                f"outputs, expected {self.num_genes}."
            )

        # First element = overall prediction.
        # Remaining elements = individual genes.
        joint_logits = torch.cat(
            [overall_logits, gene_logits],
            dim=-1,
        )

        # joint_logits:
        # [B, 1 + num_genes]

        # --------------------------------------------------
        # 5. Multi-task, multi-label loss
        # --------------------------------------------------

        loss = None
        overall_loss = None
        gene_loss = None

        if labels is not None:
            expected_shape = (
                joint_logits.shape[0],
                1 + self.num_genes,
            )

            if tuple(labels.shape) != expected_shape:
                raise ValueError(
                    f"labels must have shape {expected_shape}, "
                    f"but received {tuple(labels.shape)}."
                )

            labels = labels.float()

            overall_targets = labels[:, 0]
            gene_targets = labels[:, 1:]

            overall_loss = (
                F.binary_cross_entropy_with_logits(
                    overall_logits.squeeze(-1),
                    overall_targets,
                )
            )

            gene_loss = (
                F.binary_cross_entropy_with_logits(
                    gene_logits,
                    gene_targets,
                )
            )

            loss = (
                self.overall_loss_weight * overall_loss
                + self.gene_loss_weight * gene_loss
            )

        return {
            "loss": loss,
            "overall_loss": overall_loss,
            "gene_loss": gene_loss,
            "logits": joint_logits,
            "overall_logits": overall_logits,
            "gene_logits": gene_logits,
            "gene_embeddings": gene_embeddings,
            "patient_embeddings": patient_embeddings,
        }

# Load Data

In [11]:
!rm -rf /home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort_reader
!pip install meds_reader
!meds_reader_convert /home/jupyter/meds_sorted /home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort_reader --num_threads 32

## Filter entries table for training

In [ ]:
!pip list

In [ ]:
!/opt/conda/envs/femr-py311/bin/python3.11 -m pip uninstall femr -y
!/opt/conda/envs/femr-py311/bin/python3.11 -m pip install femr==0.1.16 datasets==2.14.5 meds==0.3.3 tqdm==4.65.0

In [ ]:
genetic_filt = pd.read_csv('/home/jupyter/workspace/data_bucket/v9_gen_data/entries_table_full_v9.csv')
genetic_filt['gene_symbols'] = genetic_filt['gene_symbols'].map(lambda x: x[2:-2])
genetic_filt.head()

In [ ]:
import json

# Acceptable ClinVar classifications
lst = [
    'pathogenic',
    'likely pathogenic',
    'likely pathogenic, pathogenic',
    'likely pathogenic, affects',
    'likely pathogenic, association',
    'likely pathogenic, drug response',
    'likely pathogenic, drug response, not provided',
    'likely pathogenic, not provided',
    'likely pathogenic, other',
    'likely pathogenic, pathogenic, affects',
    'likely pathogenic, pathogenic, association',
    'likely pathogenic, pathogenic, drug response',
    'likely pathogenic, pathogenic, drug response, not provided',
    'likely pathogenic, pathogenic, drug response, other',
    'likely pathogenic, pathogenic, drug response, protective',
    'likely pathogenic, pathogenic, drug response, risk factor',
    'likely pathogenic, pathogenic, not provided',
    'likely pathogenic, pathogenic, other',
    'likely pathogenic, pathogenic, protective',
    'likely pathogenic, pathogenic, risk factor',
    'likely pathogenic, pathogenic, risk factor, not provided',
    'likely pathogenic, risk factor',
    'likely risk allele',
    'pathogenic, association',
    'pathogenic, association, protective',
    'pathogenic, confers sensitivity',
    'pathogenic, drug response',
    'pathogenic, drug response, not provided',
    'pathogenic, drug response, risk factor',
    'pathogenic, drug response, risk factor, protective',
    'pathogenic, not provided',
    'pathogenic, other',
    'pathogenic, protective',
    'pathogenic, protective, other',
    'pathogenic, risk factor',
    'pathogenic, risk factor, not provided',
    'pathogenic, risk factor, other',
    'pathogenic, risk factor, protective',
    'risk factor'
]

# Normalize labels so spacing/capitalization differences do not matter
def normalize_classification(x):
    return ",".join(
        part.strip().lower()
        for part in x.split(",")
    )

allowed_classifications = {
    normalize_classification(x)
    for x in lst
}

# Parse exported Hail array if it was loaded from CSV as a string
genetic_filt["clinvar_classifications"] = (
    genetic_filt["clinvar_classifications"].apply(
        lambda x: json.loads(x) if isinstance(x, str) else x
    )
)

# Normalize classifications in the entries table
genetic_filt["clinvar_classifications"] = (
    genetic_filt["clinvar_classifications"].apply(
        lambda classes: [
            normalize_classification(c)
            for c in classes
        ]
    )
)

# Keep only entries whose variant has at least one EXACT acceptable classification
genetic_filt = genetic_filt[
    genetic_filt["clinvar_classifications"].apply(
        lambda classes: any(
            c in allowed_classifications
            for c in classes
        )
    )
].copy()

print("Rows remaining:", len(genetic_filt))
print("Unique patients remaining:", genetic_filt["s"].nunique())
print("Unique variants remaining:", genetic_filt["locus"].nunique())

In [ ]:
genetic_filt = genetic_filt[['s', 'gene_symbols', 'annotations']]

In [ ]:
genetic_filt['gene_symbols'].unique()

In [ ]:
json.loads(genetic_filt[genetic_filt['gene_symbols']=='MC1R']['annotations'].values[0])

In [ ]:
import json

genetic_filt["annotations"] = genetic_filt["annotations"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

print(type(genetic_filt["annotations"].iloc[0]))
print(type(genetic_filt["annotations"].iloc[0][0]))
print(genetic_filt["annotations"].iloc[0][0])

excluded_vids = {
    "PDGFRA": {"4-54281602-C-T"},
    "EGFR": {"7-55173126-T-C"},
    "RET": {
        "10-43100576-C-T",
        "10-43106497-G-A",
    },
    "TSC1": {"9-132921940-T-G"},
    "TMEM127": {"2-96265399-G-A"},
    "TERT": {
        "5-1293489-C-G",
        "5-1268581-G-A",
        "5-1254461-C-T",
    },
    "POLE": {
        "12-132680048-T-C",
        "12-132677577-C-T",
    },
    "ALK": {"2-29220747-C-T"},
}


def should_exclude(annotations):
    for ann in annotations:
        gene = ann.get("gene_symbol")
        vid = ann.get("vid")

        if gene == "MC1R":
            return True

        if gene == "EPCAM" and ann.get("variant_type") == "deletion":
            return True

        if gene in excluded_vids and vid in excluded_vids[gene]:
            return True

    return False


genetic_filt = genetic_filt[
    ~genetic_filt["annotations"].apply(should_exclude)
].copy()

In [ ]:
genetic_filt.s.nunique()

In [ ]:
genes = genetic_filt['gene_symbols'].unique()

In [ ]:
genetic_filt = genetic_filt[['s', 'gene_symbols']]

In [ ]:
genetic_filt = (
    genetic_filt[["s", "gene_symbols"]]
    .groupby("s")["gene_symbols"]
    .agg(lambda x: list(dict.fromkeys(x)))
    .reset_index()
)

genetic_filt["label"] = (
    genetic_filt["gene_symbols"]
    .map(lambda x: np.isin(genes, x).astype(int))
)

In [ ]:
genetic_filt.to_csv('/home/jupyter/workspace/data_bucket/v9_gen_data/entries_table_filt_v9.csv')

## Finalize data preprocessing

In [21]:
from pathlib import Path

dir_path = Path('/home/jupyter/workspace/data_bucket/meds/')
survey_df = pd.read_parquet('/home/jupyter/workspace/data_bucket/survey_data/survey.parquet')

In [22]:
meds_patients = set()

for file in dir_path.glob("data/*.parquet"):
    print(file)
    meds = pd.read_parquet(file)
    patients = meds.subject_id.unique()
    meds_patients.update(patients)
shared_patients = set(survey_df["person_id"]) & meds_patients

/home/jupyter/workspace/data_bucket/meds/data/0.parquet
/home/jupyter/workspace/data_bucket/meds/data/1.parquet
/home/jupyter/workspace/data_bucket/meds/data/10.parquet
/home/jupyter/workspace/data_bucket/meds/data/11.parquet
/home/jupyter/workspace/data_bucket/meds/data/12.parquet
/home/jupyter/workspace/data_bucket/meds/data/13.parquet
/home/jupyter/workspace/data_bucket/meds/data/14.parquet
/home/jupyter/workspace/data_bucket/meds/data/15.parquet
/home/jupyter/workspace/data_bucket/meds/data/16.parquet
/home/jupyter/workspace/data_bucket/meds/data/17.parquet
/home/jupyter/workspace/data_bucket/meds/data/18.parquet
/home/jupyter/workspace/data_bucket/meds/data/19.parquet
/home/jupyter/workspace/data_bucket/meds/data/2.parquet
/home/jupyter/workspace/data_bucket/meds/data/20.parquet
/home/jupyter/workspace/data_bucket/meds/data/21.parquet
/home/jupyter/workspace/data_bucket/meds/data/22.parquet
/home/jupyter/workspace/data_bucket/meds/data/23.parquet
/home/jupyter/workspace/data_bucke

KeyboardInterrupt: 

In [18]:
len(shared_patients)

0

In [ ]:
entries_v9 = pd.read_csv('/home/jupyter/workspace/data_bucket/v9_gen_data/entries_table_filt_v9.csv')
entries_v9.label

In [ ]:
entries_v9["label"] = entries_v9["label"].apply(
    lambda x: np.fromstring(x.strip("[]"), sep=" ", dtype=int)
)

In [ ]:
sum_lab = entries_v9['label'].map(lambda x: sum(x))
sum_lab.unique()

In [ ]:
survey_df = survey_df[survey_df['person_id'].isin(shared_patients)]
survey_df.to_parquet('/home/jupyter/workspace/data_bucket/survey_data/survey_filt.parquet')
entries_v9 = entries_v9[entries_v9['s'].isin(shared_patients)]
entries_v9.to_csv('/home/jupyter/workspace/data_bucket/v9_gen_data/entries_table_filt_v9.csv')

In [ ]:
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

MEDS_PATH = "/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort/data/train"

for path in Path(MEDS_PATH).rglob("*.parquet"):
    try:
        schema = pq.read_schema(path)

        null_fields = [
            field.name
            for field in schema
            if pa.types.is_null(field.type)
        ]

        if null_fields:
            print(f"\nBAD FILE: {path}")
            print("Null-typed fields:", null_fields)
            print(schema)

    except Exception as e:
        print(f"ERROR reading {path}: {e}")

In [ ]:
from pathlib import Path
import pyarrow.parquet as pq

MEDS_PATH = Path(
    "/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort/data"
)

for path in MEDS_PATH.rglob("*/*.parquet"):
    print(path)
    schema = pq.read_schema(path)

    if "text" in schema.names:
        print(f"Fixing: {path}")

        table = pq.read_table(path)
        table = table.drop(["text"])

        pq.write_table(
            table,
            path,
            compression="snappy",
        )

print("Done.")

## Load training data

In [12]:
import meds_reader
import inspect
import femr
inspect.getsource(meds_reader.SubjectDatabase)
print(femr.__version__)
database = meds_reader.SubjectDatabase(
    "/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort_reader"
)

entries_v9 = pd.read_csv('/home/jupyter/workspace/data_bucket/v9_gen_data/entries_table_filt_v9.csv')
survey_df = pd.read_parquet('/home/jupyter/workspace/data_bucket/survey_data/survey_filt.parquet')

# main_split = femr.splits.generate_hash_split(list(database), 97, frac_test=0.3)

# train_split = femr.splits.generate_hash_split(main_split.train_patient_ids, 87, frac_test=0.3)

# main_database = database.filter(main_split.train_patient_ids)
# train_database = main_database.filter(train_split.train_patient_ids)
# val_database = main_database.filter(train_split.test_patient_ids)
from sklearn.model_selection import train_test_split

subject_ids = list(database)
total_subjects = set(survey_df["person_id"]) & set(subject_ids)
print(len(total_subjects))
total_subjects = list(total_subjects)

# 70% development, 30% test
train_ids, test_ids = train_test_split(
    total_subjects,
    test_size=0.3,
    random_state=97,
)

# Split the remaining 30% into 15/15:
# 15% test, 15% validation overall
test_ids, val_ids = train_test_split(
    test_ids,
    test_size=0.5,
    random_state=87,
)

0.1.16
616567


In [13]:
for e in database[list(database)[0]].events:
    print(dict(e))

{'code': 'Ethnicity/Not Hispanic', 'table': 'person', 'time': datetime.datetime(1957, 6, 15, 0, 0)}
{'code': 'PPI/GenderIdentity_Man', 'table': 'person', 'time': datetime.datetime(1957, 6, 15, 0, 0)}
{'code': 'PPI/WhatRaceEthnicity_Black', 'table': 'person', 'time': datetime.datetime(1957, 6, 15, 0, 0)}
{'code': 'MEDS_BIRTH', 'table': 'person', 'time': datetime.datetime(1957, 6, 15, 0, 0)}
{'code': 'CPT4/90792', 'table': 'procedure', 'time': datetime.datetime(2013, 3, 26, 0, 0), 'visit_id': 34000000000597018}
{'code': 'ICD9CM/296.33', 'table': 'condition', 'time': datetime.datetime(2013, 3, 26, 5, 0), 'visit_id': 34000000000597018}
{'code': 'ICD9CM/294.8', 'table': 'condition', 'time': datetime.datetime(2013, 3, 26, 5, 0), 'visit_id': 34000000000597018}
{'code': 'Visit/OP', 'end': datetime.datetime(2013, 3, 26, 23, 59, 59), 'table': 'visit', 'time': datetime.datetime(2013, 3, 26, 13, 15), 'visit_id': 34000000000597018}
{'code': 'Visit/OP', 'end': datetime.datetime(2013, 3, 26, 23, 59, 

In [ ]:
tokenizer.code_2_token

# Train Model